In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab";

The function $\texttt{toDot}(\texttt{Parent})$ takes a dictionary $\texttt{Parent}$.
For every node $x$, $\texttt{Parent}[x]$ is the parent of $x$.   It draws this dictionary 
as a family tree using `Graphviz`, i.e. for every node $x$ it draws an arrow starting at $x$ and pointing 
to $\texttt{Parent}[x]$.  The roots of the trees are indicated by double circles.

In [ ]:
const gv = await Graphviz.load();

function toDot(Parent: Record<number, number>): string {
  let dot = "digraph G {\n";
  for (const x in Parent) {
    const p = Parent[x];
    if (Number(x) === p) {
      dot += `  "${x}" [shape=doublecircle];\n`;
    } else {
      dot += `  "${x}" [shape=circle];\n`;
      dot += `  "${x}" -> "${p}";\n`;
    }
  }
  dot += "}";
  return dot;
}

## A Tree Based Implementation of the Union-Find Algorithm

Given a dictionary `Parent` and an element $x$ from $M$, the function $\texttt{find}(x, \texttt{Parent})$ 
returns the ancestor of $x$ that is its own parent.

In [ ]:
function find(x: number, Parent: Record<number, number>): number {
  const p = Parent[x];
  return p === x ? x : find(p, Parent);
}

Given a set $M$ and a binary relation $R \subseteq M \times M$, the function $\texttt{union\_find}$ returns a *partition* $\mathcal{P}$ of $M$ such that we have
$$ \forall \langle x, y \rangle \in R: \exists S \in \mathcal{P}: \bigl(x \in S \wedge y \in S\bigr) $$
The resulting partition defines the equivalence relation that is generated by $R$.

In [ ]:
function union_find(M: number[], R: [number, number][]): number[][] {
  const Parent: Record<number, number> = {};
  for (const x of M) Parent[x] = x;

  for (const [x, y] of R) {
    console.log(`${x} ≅ ${y}`);
    const root_x = find(x, Parent);
    const root_y = find(y, Parent);

    {
      const dot = toDot(Parent);
      const svg = gv.layout(dot, "svg", "dot");
      display.html(svg);
    }

    if (root_x !== root_y) {
      Parent[root_y] = root_x;

      const dot = toDot(Parent);
      const svg = gv.layout(dot, "svg", "dot");
      display.html(svg);
    }
  }

  const Roots = M.filter(x => Parent[x] === x);
  return Roots.map(r => M.filter(y => find(y, Parent) === r));
}

In [ ]:
function demo() {
  const M = Array.from({ length: 9 }, (_, i) => i + 1);
  const R: [number, number][] = [
    [1, 4],
    [7, 9],
    [3, 5],
    [2, 6],
    [5, 8],
    [1, 9],
    [4, 7],
  ];
  const P = union_find(M, R);
  return P;
}

In [ ]:
demo();

In [ ]:
function worst_case(n: number) {
  const M = Array.from({ length: n }, (_, i) => i + 1);
  const R: [number, number][] = [];
  for (let k = 1; k < n; k++) {
    R.push([k + 1, k]);
  }
  console.log("R =", R);
  const P = union_find(M, R);
  console.log("P =", P);
}

In [ ]:
worst_case(10);

The previous example was a worst case scenario because we had defined the relation $R$ as an array so that we were able to control the order of the joining of different trees. If we represent $R$ as a set, the order of the pairs is more or less random and the trees do no degenerate to arrays.

In [ ]:
function shuffle<T>(A: T[]): T[] {
  for (let i = A.length - 1; i > 0; i--) {
    const j = Math.floor(Math.random() * (i + 1));
    [A[i], A[j]] = [A[j], A[i]];
  }
  return A;
}

In [ ]:
function worst_case_set(n: number) {
  const M = Array.from({ length: n }, (_, i) => i + 1);
  const R: [number, number][] = [];
  for (let k = 1; k < n; k++) R.push([k + 1, k]);
  shuffle(R);
  console.log("R (shuffled) =", R);
  const P = union_find(M, R);
  console.log("P =", P);
}

In [ ]:
worst_case_set(20);